# Building Nested Domains with CstarSpecEngine

This notebook demonstrates how to use `CstarSpecEngine` to build nested domains.


## Setup

Enable autoreload for development and import the package.


In [ ]:
%load_ext autoreload
%autoreload 2

## Generate All Domains

Create a `CstarSpecEngine` instance and generate all domains from `domains.yml`.

**Parameters:**
- `clobber_inputs`: If `True`, overwrite existing input files for all domains
- `clobber_source_data`: If `True`, re-download source datasets
- `partition_files`: If `True`, partition input files across tiles
- `test`: If `True`, truncate generation loop after 2 iterations (for testing)
- `compile_time_settings`: Optional dict of compile-time settings overrides
- `run_time_settings`: Optional dict of run-time settings overrides
- `overrides`: Optional dict of configuration overrides to apply to all domains

The method returns a dictionary mapping domain names to `CstarSpecBuilder` instances.


In [ ]:
import cson_forge

## Run All Simulations

Execute model simulations for all generated domains. The `run_all()` method handles execution for all builders and returns execution handlers for monitoring the runs.

**Note:** This cell is commented out by default. Uncomment to run all simulations.


In [ ]:
%time 
engine = cson_forge.CstarSpecEngine(domains_file="PAC_domains-nested.yml", )
builders = engine.generate_all(clobber_inputs=False, 
                               stop_on_failure=False, 
                               )

## This should be runnable with run_time_settings={"roms.in": {"time_stepping": {"dt":900}}} according to documentation
## but this will actually set all run_time_settings to null except whatever is explicity called.... 
## for now I will update the time step after setup

In [9]:
builders.keys()

dict_keys(['GoA-p10_deg', 'NEP-p25_deg', 'PAC-0p5_deg'])

## Manual changes:
- change time stepping dt
- change name and location of nesting file

In [ ]:
key = 

In [ ]:
## TIME STEP
import glob 
import yaml

bp_path = glob.glob(str(builders[key].blueprint_dir) + '/B_*_build.yml')[0]
with open(bp_path, "r") as f:
    config = yaml.safe_load(f)

## manually change timestep --- this shouldn't be needed in the future
config["model_params"]["time_step"] = 900
print(config)

# ## overwrite blueprint in place
with open(bp_path, "w") as f:
    yaml.dump(config, f, sort_keys=False)

{'name': 'cson_roms-marbl_v0.1_GoA-p10_deg_256procs', 'description': 'Gulf of Alaska 1/10 deg (nested)', 'application': 'roms_marbl', 'state': 'notset', 'valid_start_date': '2010-01-03T00:00:00', 'valid_end_date': '2010-01-10T00:00:00', 'code': {'roms': {'documentation': '', 'locked': False, 'location': 'https://github.com/CWorthy-ocean/ucla-roms.git', 'commit': '00706a1', 'branch': '', 'filter': None}, 'run_time': {'documentation': '', 'locked': False, 'location': '/anvil/projects/x-ees250129/x-awyatt1/cson-forge/cson_forge/builds/cson_roms-marbl_v0.1_GoA-p10_deg_256procs/run-time', 'commit': '', 'branch': 'na', 'filter': {'files': ['marbl_diagnostic_output_list', 'marbl_in', 'marbl_tracer_output_list', 'roms.in']}}, 'compile_time': {'documentation': '', 'locked': False, 'location': '/anvil/projects/x-ees250129/x-awyatt1/cson-forge/cson_forge/builds/cson_roms-marbl_v0.1_GoA-p10_deg_256procs/compile-time', 'commit': '', 'branch': 'na', 'filter': {'files': ['Makefile', 'bgc.opt', 'blk_f

In [14]:
bp_path

'/anvil/projects/x-ees250129/x-awyatt1/cson-forge/cson_forge/blueprints/RCAC_anvil/cson_roms-marbl_v0.1_GoA-p10_deg_256procs/B_cson_roms-marbl_v0.1_GoA-p10_deg_256procs_build.yml'

In [12]:
## NESTING FILE

import shutil
import os
import re

for key in ['PAC-0p5_deg','NEP-p25_deg']:
    opt_file = str(builders[key].compile_time_code_dir) + '/extract_data.opt'
    with open(opt_file) as f:
        text = f.read()
    
    # get path of the nesting file from extract_data.opt
    block = text.split('extract_file')[1]
    old_path = ''.join(re.findall(r"'([^']*)'", block))
    
    print("Old path:", old_path)
    
    # Copy nesting file to short named directory
    dst_dir = '/anvil/projects/x-ees250129/'
    
    filename = os.path.basename(old_path)
    new_name = filename.replace('cson_roms-marbl_v0.1_', '')
    dst = os.path.join(dst_dir, new_name)
    
    new_nc_file = shutil.copy(old_path, dst)
    print("New nesting file path:", new_nc_file)
    
    # replace filepath in extract_data.opt
    with open(opt_file, 'r') as f:
        lines = f.readlines()
    
    new_lines = []
    i = 0
    
    while i < len(lines):
        line = lines[i]
    
        if 'extract_file' in line and '=' in line:
            # skip original multi-line definition
            i += 1
            while i < len(lines) and ("'" in lines[i] or '//' in lines[i] or '&' in lines[i]):
                i += 1
    
            # insert clean single-line version
            new_lines.append(f"      character(len=512) :: extract_file = '{new_nc_file}'\n")
    
        else:
            new_lines.append(line)
            i += 1
    
    with open(opt_file, 'w') as f:
        f.writelines(new_lines)

print("done ammending files")

Old path: /anvil/projects/x-ees250129/PAC-0p5_deg_256procs_nesting.nc


SameFileError: '/anvil/projects/x-ees250129/PAC-0p5_deg_256procs_nesting.nc' and '/anvil/projects/x-ees250129/PAC-0p5_deg_256procs_nesting.nc' are the same file

### Move to CLI and run in CSTAR

You can now run the `B_*_build.yml` blueprint using Cstar, after which, the files will need to be post-processed for nesting. You can do this currently using `nesting_adapt_files_fr_parent.ipynb` here: 

https://github.com/CWorthy-ocean/internal_roms_scripts/blob/ff5ab569a873c3f4ed5556835e752adb4a50a0a0/Abigale/nesting_adapt_files_fr_parent.ipynb